# Load data from tensorflow datasets

In [19]:
!pip install tensorflow_datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.3 MB 1.2 MB/s eta 0:00:05
   ------- -------------------------------- 1.0/5.3 MB 1.7 MB/s eta 0:00:03
   ------------- -------------------------- 1.8/5.3 MB 2.2 MB/s eta 0:00:02
   ------------------- -------------------- 2.6/5.3 MB 2.6 MB/s eta 0:00:02
   ------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
centrifuge-python 0.4.0 requires protobuf<6.0.0,>=4.23.4, but you have protobuf 6.32.0 which is incompatible.
streamlit 1.41.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.0 which is incompatible.
tensorflow-intel 2.13.1 requires numpy<=1.24.3,>=1.22, but you have numpy 1.26.4 which is incompatible.
tensorflow-intel 2.13.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.32.0 which is incompatible.
tensorflow-intel 2.13.1 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.14.0 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

# Load the dataset (split into train, validation, and test)
# as_supervised=True gives us (image, label) tuples automatically
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    'oxford_flowers102',
    split=['train', 'validation', 'test'],
    as_supervised=True,
    with_info=True
)

# Check the number of classes (should be 102)
num_classes = ds_info.features['label'].num_classes

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

In [14]:
def preprocess_img(image, label):
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]
    return image, label

# Apply preprocessing, shuffle, and batch
BATCH_SIZE = 32

train_batches = (ds_train
                 .map(preprocess_img, num_parallel_calls=tf.data.AUTOTUNE)
                 .cache()
                 .shuffle(1000)
                 .batch(BATCH_SIZE)
                 .prefetch(tf.data.AUTOTUNE))

val_batches = (ds_val
               .map(preprocess_img, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(BATCH_SIZE)
               .cache())

{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNX86, Created on: Thu Feb 19 17:38:58 2009',
 '__version__': '1.0',
 '__globals__': [],
 'trnid': array([[6765, 6755, 6768, ..., 8026, 8036, 8041]], dtype=uint16),
 'valid': array([[6773, 6767, 6739, ..., 8028, 8008, 8030]], dtype=uint16),
 'tstid': array([[6734, 6735, 6737, ..., 8044, 8045, 8047]], dtype=uint16)}

# Load data from local

In [3]:
import scipy.io
import os
import numpy as np
import tensorflow as tf

# Load the .mat file
mat_data = scipy.io.loadmat('./flower/labels.mat')

# Extract the labels (Update 'labels' to match your specific key)
# Ensure the labels are 0-indexed for CrossEntropy
labels = mat_data['labels'].flatten().astype('int32') 

# Get a sorted list of image file paths to match the label order
img_dir = './flower/images/'
img_paths = [os.path.join(img_dir, f) for f in sorted(os.listdir(img_dir))]

In [15]:
mat_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: GLNX86, Created on: Thu Feb 19 15:43:33 2009',
 '__version__': '1.0',
 '__globals__': [],
 'labels': array([[77, 77, 77, ..., 62, 62, 62]], dtype=uint8)}

In [6]:
img_paths

['./flower/images/image_00001.jpg',
 './flower/images/image_00002.jpg',
 './flower/images/image_00003.jpg',
 './flower/images/image_00004.jpg',
 './flower/images/image_00005.jpg',
 './flower/images/image_00006.jpg',
 './flower/images/image_00007.jpg',
 './flower/images/image_00008.jpg',
 './flower/images/image_00009.jpg',
 './flower/images/image_00010.jpg',
 './flower/images/image_00011.jpg',
 './flower/images/image_00012.jpg',
 './flower/images/image_00013.jpg',
 './flower/images/image_00014.jpg',
 './flower/images/image_00015.jpg',
 './flower/images/image_00016.jpg',
 './flower/images/image_00017.jpg',
 './flower/images/image_00018.jpg',
 './flower/images/image_00019.jpg',
 './flower/images/image_00020.jpg',
 './flower/images/image_00021.jpg',
 './flower/images/image_00022.jpg',
 './flower/images/image_00023.jpg',
 './flower/images/image_00024.jpg',
 './flower/images/image_00025.jpg',
 './flower/images/image_00026.jpg',
 './flower/images/image_00027.jpg',
 './flower/images/image_0002

In [10]:
def load_and_preprocess_image(path, label):
    # Read image file
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    # Resize and rescale (normalize to [0,1])
    image = tf.image.resize(image, [224, 224])
    image = image / 255.0
    return image, label -1

# Create the dataset object
dataset = tf.data.Dataset.from_tensor_slices((img_paths, labels))
dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Shuffle and batch
BATCH_SIZE = 32
dataset = dataset.shuffle(buffer_size=len(img_paths)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [8]:
model = tf.keras.Sequential([
    # Convolutional layers
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    # Flatten and Dense layers
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5), # Prevents overfitting
    tf.keras.layers.Dense(len(np.unique(labels)), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
# Train the model
model.fit(dataset, epochs=10)

# Evaluate the model
loss, accuracy = model.evaluate(dataset)
print(f"Final Accuracy: {accuracy * 100:.2f}%")

Epoch 1/10
186/256 [====================>.........] - ETA: 57s - loss: 4.5866 - accuracy: 0.0410

KeyboardInterrupt: 